# UKF–SIR Narrative Model
## Day 2 — Fake Data Test (Single Observable: Google Trends)

**Goal:** simulate a SIR narrative trajectory with known parameters, generate noisy GT observations, run MLE, and verify parameter recovery.

**State-space model recap**

$$S_{t+1} = S_t - \beta S_t I_t + w_S, \qquad I_{t+1} = I_t + (\beta S_t I_t - \gamma I_t) + w_I, \qquad \mathbf{w}_t\sim\mathcal{N}(\mathbf{0},Q)$$

$$y_t^{GT} = I_t + v_t, \qquad v_t\sim\mathcal{N}(0,\sigma^2_{\varepsilon_1}), \qquad c_1=1 \text{ fixed}$$

**Free parameters:** $\theta = [\beta,\; \gamma,\; \sigma^2_S,\; \sigma^2_I,\; \sigma^2_{\varepsilon_1},\; I_0]$

---
## Day 1 Infrastructure

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── 1. SIR TRANSITION (Euler discretisation) ─────────────────────────────────

def sir_transition(x, beta, gamma, dt=1.0):
    """
    x = [S, I] as fractions in [0,1], S+I+R=1.
    Returns x_next = [S_next, I_next].
    """
    S, I = x[0], x[1]
    S_next = S - beta * S * I * dt
    I_next = I + (beta * S * I - gamma * I) * dt
    return np.array([S_next, I_next])

In [ ]:
# ── 2. MERWE SCALED SIGMA POINTS ─────────────────────────────────────────────

def sigma_points(x, P, alpha=1e-3, beta_p=2.0, kappa=0.0):
    """
    Returns:
        sigmas : (2n+1, n)
        Wm     : (2n+1,)  mean weights
        Wc     : (2n+1,)  covariance weights
    """
    n   = len(x)
    lam = alpha**2 * (n + kappa) - n
    c   = n + lam

    Wm    = np.full(2*n + 1, 0.5 / c)
    Wm[0] = lam / c
    Wc    = Wm.copy()
    Wc[0] += (1.0 - alpha**2 + beta_p)

    try:
        L = np.linalg.cholesky(c * P)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(c * P + 1e-10 * np.eye(n))

    sigmas        = np.empty((2*n + 1, n))
    sigmas[0]     = x
    for i in range(n):
        sigmas[i + 1]     = x + L[:, i]
        sigmas[n + i + 1] = x - L[:, i]

    return sigmas, Wm, Wc

In [ ]:
# ── 3. UKF (2D state, 1D observation) ────────────────────────────────────────

class UKF1D:
    """
    State  : [S, I]  (2-D)
    Obs    : GT      (scalar, normalised to [0,1] by dividing raw GT by 100)
    c1 = 1 fixed  →  h(x) = I
    """

    def __init__(self, beta, gamma, Q, R_obs, x0, P0, dt=1.0,
                 alpha=1e-3, beta_ukf=2.0, kappa=0.0):
        self.beta     = beta
        self.gamma    = gamma
        self.Q        = np.asarray(Q, dtype=float)
        self.R_obs    = float(R_obs)
        self.x        = np.asarray(x0, dtype=float).copy()
        self.P        = np.asarray(P0, dtype=float).copy()
        self.dt       = dt
        self.alpha    = alpha
        self.beta_ukf = beta_ukf
        self.kappa    = kappa

    def _f(self, x):
        return sir_transition(x, self.beta, self.gamma, self.dt)

    def _h(self, x):
        return x[1]                          # I(t), c1 = 1

    def predict(self):
        sp, Wm, Wc = sigma_points(self.x, self.P,
                                   self.alpha, self.beta_ukf, self.kappa)
        sp_f   = np.array([self._f(s) for s in sp])
        x_pred = Wm @ sp_f
        diff   = sp_f - x_pred
        P_pred = np.einsum('i,ij,ik->jk', Wc, diff, diff) + self.Q
        self.x, self.P = x_pred, P_pred

    def update(self, y_obs):
        """
        Returns
        -------
        nu   : scalar innovation
        S_yy : scalar innovation variance
        """
        sp, Wm, Wc = sigma_points(self.x, self.P,
                                   self.alpha, self.beta_ukf, self.kappa)
        sp_h   = np.array([self._h(s) for s in sp])
        y_pred = Wm @ sp_h

        diff_y = sp_h - y_pred
        diff_x = sp   - self.x

        S_yy = np.dot(Wc, diff_y**2) + self.R_obs
        P_xy = np.einsum('i,ij,i->j', Wc, diff_x, diff_y)

        K    = P_xy / S_yy
        nu   = y_obs - y_pred

        self.x = self.x + K * nu
        self.P = self.P - np.outer(K, K) * S_yy

        return nu, S_yy

    def step(self, y_obs):
        self.predict()
        return self.update(y_obs)

    def run(self, observations):
        """
        observations : (T,) normalised GT in [0,1]

        Returns
        -------
        states      : (T, 2)  filtered [S, I]
        innovations : (T,)    innovation sequence  ν(t)
        inn_vars    : (T,)    innovation variance   S_yy(t)
        """
        T           = len(observations)
        states      = np.empty((T, 2))
        innovations = np.empty(T)
        inn_vars    = np.empty(T)

        for t in range(T):
            nu, S_yy        = self.step(observations[t])
            states[t]       = self.x
            innovations[t]  = nu
            inn_vars[t]     = S_yy

        return states, innovations, inn_vars

In [ ]:
# ── 4. LOG-LIKELIHOOD ─────────────────────────────────────────────────────────

def ukf_negloglik(params, observations, dt=1.0):
    """
    params = [log_β, log_γ, log_σ²_S, log_σ²_I, log_σ²_ε1, logit_I0]

    Positivity / box constraints enforced via log / logit transforms.
    Returns negative log-likelihood (scalar, for minimisation).
    """
    log_b, log_g, log_s2S, log_s2I, log_s2e, logit_I0 = params

    beta        = np.exp(log_b)
    gamma       = np.exp(log_g)
    sigma2_S    = np.exp(log_s2S)
    sigma2_I    = np.exp(log_s2I)
    sigma2_eps1 = np.exp(log_s2e)
    I0          = 1.0 / (1.0 + np.exp(-logit_I0))
    S0          = 1.0 - I0

    Q   = np.diag([sigma2_S, sigma2_I])
    P0  = np.diag([1e-6, 1e-6])
    x0  = np.array([S0, I0])

    ukf     = UKF1D(beta, gamma, Q, sigma2_eps1, x0, P0, dt)
    log_lik = 0.0

    for y in observations:
        ukf.predict()
        nu, S_yy = ukf.update(y)

        if S_yy <= 0 or not np.isfinite(S_yy):
            return 1e12

        log_lik += -0.5 * (np.log(2 * np.pi * S_yy) + nu**2 / S_yy)

    return -log_lik


# ── 5. OPTIMIZER WRAPPER ──────────────────────────────────────────────────────

def estimate_parameters(observations, dt=1.0, n_restarts=5, verbose=True):
    """
    MLE via L-BFGS-B with n_restarts random starts.

    observations : (T,) GT values normalised to [0,1]

    Returns
    -------
    best_params : dict
    best_result : scipy OptimizeResult
    """
    obs = np.asarray(observations, dtype=float)

    bounds = [
        (-5.0,  1.0),   # log β
        (-5.0,  1.0),   # log γ
        (-14.0, -2.0),  # log σ²_S
        (-14.0, -2.0),  # log σ²_I
        (-10.0,  2.0),  # log σ²_ε1
        (-8.0,  -0.5),  # logit I0
    ]

    x0_default = np.array([
        np.log(0.30), np.log(0.10),
        np.log(1e-5), np.log(1e-5),
        np.log(5e-3),
        np.log(0.01 / 0.99),
    ])

    best_val, best_result = np.inf, None
    rng_opt = np.random.default_rng(seed=0)

    for i in range(n_restarts):
        x0 = x0_default if i == 0 else np.array(
            [rng_opt.uniform(lo, hi) for lo, hi in bounds]
        )
        try:
            res = minimize(
                ukf_negloglik, x0,
                args=(obs, dt),
                method='L-BFGS-B',
                bounds=bounds,
                options={'maxiter': 2000, 'ftol': 1e-10, 'gtol': 1e-7},
            )
            if verbose:
                print(f'  restart {i}: negloglik = {res.fun:.4f}  success={res.success}')
            if res.fun < best_val:
                best_val, best_result = res.fun, res
        except Exception as e:
            if verbose:
                print(f'  restart {i}: FAILED ({e})')

    p  = best_result.x
    I0 = 1.0 / (1.0 + np.exp(-p[5]))

    best_params = {
        'beta':        np.exp(p[0]),
        'gamma':       np.exp(p[1]),
        'R0':          np.exp(p[0]) / np.exp(p[1]),
        'sigma2_S':    np.exp(p[2]),
        'sigma2_I':    np.exp(p[3]),
        'sigma2_eps1': np.exp(p[4]),
        'I0':          I0,
        'S0':          1.0 - I0,
        'negloglik':   best_val,
    }
    return best_params, best_result

---
## Day 2 — Fake Data Test

### Step 1 — Simulate ground truth

In [ ]:
rng = np.random.default_rng(seed=42)

# ── True parameters ──────────────────────────────────────────────────────────
BETA_TRUE  = 0.30
GAMMA_TRUE = 0.10
S2_S_TRUE  = 1e-5
S2_I_TRUE  = 1e-5
S2_EPS_TRUE = 5e-3
I0_TRUE    = 0.01
T          = 52          # 52 weekly observations

print('True parameters')
print(f'  β   = {BETA_TRUE}   γ   = {GAMMA_TRUE}   R₀ = {BETA_TRUE/GAMMA_TRUE:.1f}')
print(f'  I₀  = {I0_TRUE}   σ²_ε1 = {S2_EPS_TRUE}')

In [ ]:
# ── Simulate S(t), I(t) with process noise ────────────────────────────────────
S_true = np.empty(T)
I_true = np.empty(T)

S, I = 1.0 - I0_TRUE, I0_TRUE
for t in range(T):
    S_true[t] = S
    I_true[t] = I
    wS = rng.normal(0, np.sqrt(S2_S_TRUE))
    wI = rng.normal(0, np.sqrt(S2_I_TRUE))
    S_next = S - BETA_TRUE * S * I + wS
    I_next = I + (BETA_TRUE * S * I - GAMMA_TRUE * I) + wI
    S = np.clip(S_next, 0.0, 1.0)
    I = np.clip(I_next, 0.0, 1.0 - S)

# ── Generate noisy GT observations (normalised to [0,1]) ─────────────────────
GT_obs = np.clip(
    I_true + rng.normal(0, np.sqrt(S2_EPS_TRUE), T),
    0.0, 1.0
)

weeks = np.arange(T)
print(f'I_true range : [{I_true.min():.4f}, {I_true.max():.4f}]')
print(f'GT_obs range : [{GT_obs.min():.4f}, {GT_obs.max():.4f}]')

### Step 2 — Filter sanity check with true parameters

In [ ]:
ukf_true = UKF1D(
    beta=BETA_TRUE, gamma=GAMMA_TRUE,
    Q=np.diag([S2_S_TRUE, S2_I_TRUE]),
    R_obs=S2_EPS_TRUE,
    x0=np.array([1.0 - I0_TRUE, I0_TRUE]),
    P0=np.diag([1e-6, 1e-6]),
)
states_true, innov_true, innvar_true = ukf_true.run(GT_obs)
std_innov_true = innov_true / np.sqrt(innvar_true)

print('Filter with TRUE params')
print(f'  Innovation mean : {innov_true.mean():+.5f}  (expect ≈ 0)')
print(f'  Std(ν/√S_yy)    : {std_innov_true.std():.4f}   (expect ≈ 1)')

### Step 3 — MLE parameter recovery

In [ ]:
print('Running MLE (L-BFGS-B, 5 restarts) …\n')
est_params, opt_result = estimate_parameters(GT_obs, dt=1.0, n_restarts=5, verbose=True)

print('\n=== Recovery ===')
print(f"{'Parameter':<12} {'True':>8} {'Estimated':>10} {'Rel. err':>9}")
print('-' * 44)
checks = [
    ('β',         BETA_TRUE,    est_params['beta']),
    ('γ',         GAMMA_TRUE,   est_params['gamma']),
    ('R₀',        BETA_TRUE/GAMMA_TRUE, est_params['R0']),
    ('I₀',        I0_TRUE,      est_params['I0']),
    ('σ²_ε1',     S2_EPS_TRUE,  est_params['sigma2_eps1']),
]
for name, true_v, est_v in checks:
    err = abs(est_v - true_v) / true_v * 100
    print(f"  {name:<10} {true_v:>8.5f} {est_v:>10.5f} {err:>8.1f}%")
print(f'\nNeg log-likelihood : {est_params["negloglik"]:.3f}')
print(f'Optimizer success  : {opt_result.success}')

### Step 4 — Re-filter with estimated parameters

In [ ]:
ukf_est = UKF1D(
    beta=est_params['beta'],
    gamma=est_params['gamma'],
    Q=np.diag([est_params['sigma2_S'], est_params['sigma2_I']]),
    R_obs=est_params['sigma2_eps1'],
    x0=np.array([est_params['S0'], est_params['I0']]),
    P0=np.diag([1e-6, 1e-6]),
)
states_est, innov_est, innvar_est = ukf_est.run(GT_obs)
std_innov_est = innov_est / np.sqrt(innvar_est)

print('Filter with ESTIMATED params')
print(f'  Innovation mean : {innov_est.mean():+.5f}  (expect ≈ 0)')
print(f'  Std(ν/√S_yy)    : {std_innov_est.std():.4f}   (expect ≈ 1)')

### Step 5 — Diagnostic plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(
    f'Day 2 — UKF Parameter Recovery\n'
    f'True: β={BETA_TRUE}, γ={GAMMA_TRUE}   →   '
    f'Est: β={est_params["beta"]:.3f}, γ={est_params["gamma"]:.3f}',
    fontsize=12, fontweight='bold'
)

# ─ I(t): true vs filter (true params) vs filter (est params) ─
ax = axes[0, 0]
ax.scatter(weeks, GT_obs, s=14, c='grey', alpha=0.55, zorder=1, label='GT obs')
ax.plot(weeks, I_true,           'k-',  lw=2.0, zorder=2, label='True I(t)')
ax.plot(weeks, states_true[:, 1], 'b--', lw=1.5, zorder=3, label='UKF (true θ)')
ax.plot(weeks, states_est[:, 1],  'r:',  lw=1.8, zorder=4, label='UKF (est θ)')
ax.set(xlabel='Week', ylabel='I (normalised)', title='Infected / Narrative-Engaged')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ S(t) ─
ax = axes[0, 1]
ax.plot(weeks, S_true,           'k-',  lw=2.0, label='True S(t)')
ax.plot(weeks, states_true[:, 0], 'b--', lw=1.5, label='UKF (true θ)')
ax.plot(weeks, states_est[:, 0],  'r:',  lw=1.8, label='UKF (est θ)')
ax.set(xlabel='Week', ylabel='S (normalised)', title='Susceptible')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ Standardised innovations (estimated params) ─
ax = axes[1, 0]
ax.plot(weeks, std_innov_est, 'k-', lw=0.9, label='Std. innovation')
ax.axhline(0,     color='red',    ls='--', lw=1.0)
ax.axhline( 1.96, color='orange', ls=':',  lw=1.0, label='±1.96')
ax.axhline(-1.96, color='orange', ls=':',  lw=1.0)
ax.set(xlabel='Week', ylabel='ν / √S_yy',
       title='Standardised Innovations — est. params (should ≈ N(0,1))')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ─ Parameter bar chart ─
ax = axes[1, 1]
labels = ['β', 'γ', 'R₀']
true_v = [BETA_TRUE, GAMMA_TRUE, BETA_TRUE/GAMMA_TRUE]
est_v  = [est_params['beta'], est_params['gamma'], est_params['R0']]
x_pos  = np.arange(len(labels))
b1 = ax.bar(x_pos - 0.2, true_v, 0.35, label='True',      color='steelblue', alpha=0.8)
b2 = ax.bar(x_pos + 0.2, est_v,  0.35, label='Estimated', color='tomato',    alpha=0.8)
for bar, tv, ev in zip(b2, true_v, est_v):
    err = abs(ev - tv) / tv * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{err:.1f}%', ha='center', va='bottom', fontsize=9, color='tomato')
ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=11)
ax.set(ylabel='Value', title='Parameter Recovery  (% = relative error)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('day2_ukf_recovery.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → day2_ukf_recovery.png')

---
### Summary

| Parameter | True | Estimated |
|-----------|------|-----------|
| β (spread rate) | 0.30 | see output |
| γ (recovery rate) | 0.10 | see output |
| R₀ = β/γ | 3.0 | see output |
| I₀ | 0.01 | see output |

**Checks before Day 3:**
- β, γ recovered within ~5 % of truth.
- Standardised innovations have mean ≈ 0, std ≈ 1.

**Next (Day 3):** extend observation equation to Articles and Tone with free $c_2$, $c_3 < 0$; add diagonal 3×3 $R$; implement observation mask for days where GDELT data is missing.